In [22]:
import pandas as pd

url = "https://raw.githubusercontent.com/ankita1112/House-Prices-Advanced-Regression/master/train.csv"

df = pd.read_csv(url)

print(df.shape)
df.head()


(1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [63]:
df["TotalBath"] = (
    df["FullBath"] +
    0.5 * df["HalfBath"] +
    df["BsmtFullBath"] +
    0.5 * df["BsmtHalfBath"]
)
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
df["HasPool"] = (df["PoolArea"] > 0).astype(int)
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

In [70]:
# log target is importance for xgboost good for RMSE
import numpy as np
y = np.log1p(df["SalePrice"])
X = df.drop(['SalePrice', 'Id', 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF'], axis=1)

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(include=["float64, int64"]).columns

In [71]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 78 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   Alley          91 non-null     object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Neighborhood   1460 non-null   object 
 12  Condition1     1460 non-null   object 
 13  Condition2     1460 non-null   object 
 14  BldgType       1460 non-null   object 
 15  HouseStyle     1460 non-null   object 
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuil

In [72]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [73]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=1))
])

In [74]:
from sklearn.model_selection import cross_val_score, KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X, y, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1)
rmse = -scores
print("Fold RMSE:", rmse)
print("Mean RMSE:", rmse.mean())

Fold RMSE: [0.19404991 0.19807015 0.18385291 0.21237028 0.17061217]
Mean RMSE: 0.1917910848818548


In [75]:
from sklearn.model_selection import GridSearchCV, KFold
param_grid = {
    "model__n_estimators": [400, 600],
    "model__max_depth": [20, 40],
    "model__min_samples_leaf": [2, 4],
    "model__max_features": ["sqrt", 0.5],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print("Best RMSE", -grid.best_score_)
print("Best params", grid.best_params_)


Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best RMSE 0.19280538000829658
Best params {'model__max_depth': 40, 'model__max_features': 0.5, 'model__min_samples_leaf': 2, 'model__n_estimators': 400}


In [76]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
param_grid = {
    "model__n_estimators": [1200, 2000],
    "model__learning_rate": [0.02, 0.03],
    "model__max_depth": [3, 4],
    "model__subsample": [0.8, 0.9],
}
pipeline2 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor())
])

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

preds = grid.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, preds))
print("RMSE:", rmse)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
RMSE: 0.17091131640717475


In [78]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipeline2 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor())
])
scores = cross_val_score(pipeline2, X, y, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1)
rmse = -scores
print("Fold RMSE:", rmse)
print("Mean RMSE:", rmse.mean())

Fold RMSE: [0.19049276 0.20435691 0.18866037 0.19798432 0.1791304 ]
Mean RMSE: 0.19212495286767908
